# Reconocimiento de placas en via de lastreEjecuta las celdas **en orden, de arriba a abajo**. Cada una dice que hace.Solo hay una pausa obligatoria: la celda 3 pide reiniciar el entorno. Ahi tedetienes, reinicias, y sigues desde la celda 4.**Antes de empezar:** activa la GPU en `Entorno de ejecucion > Cambiar tipo deentorno de ejecucion > GPU`. Si la cambias despues, Colab se reinicia y pierdeslo avanzado.

## 1. Comprobar que hay GPU asignada

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv!nvcc --version | grep release

## 2. Montar tu Google DriveAprueba la ventana emergente. Los videos y los resultados viven aqui: el discode Colab se borra al desconectar.

In [2]:
from google.colab import drivedrive.mount('/content/drive')

SyntaxError: invalid syntax (909229373.py, line 1)

## 3. Instalar dependencias`onnxruntime-gpu` del indice normal de pip exige CUDA 13, pero Colab traeCUDA 12. Por eso se instala desde el repositorio de Microsoft para CUDA 12,que es el unico que funciona aqui.**Al terminar esta celda, reinicia el entorno:** `Entorno de ejecucion >Reiniciar entorno de ejecucion`. Sin reiniciar, Python sigue con la libreriavieja cargada. Luego continua desde la celda 4.

In [ ]:
!pip install -q opencv-python-headless numpy openpyxl pillow!pip install -q fast-alpr open-image-models!pip uninstall -y -q onnxruntime onnxruntime-gpu# Repositorio de Microsoft con las compilaciones para CUDA 12!pip install -q onnxruntime-gpu \    --index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ \    --extra-index-url https://pypi.org/simpleprint()print("=" * 60)print("AHORA REINICIA EL ENTORNO DE EJECUCION Y SIGUE EN LA CELDA 4")print("=" * 60)

## 4. Traer el codigoSe puede ejecutar tantas veces como quieras: si ya existe, solo lo actualiza.

In [ ]:
import osif os.path.isdir('/content/lastre/.git'):    %cd /content/lastre    !git pull --quietelse:    !rm -rf /content/lastre    !git clone --quiet https://github.com/riofutabac/PLACAS-RECONOCIMIENTO.git /content/lastre    %cd /content/lastre!git log --oneline -1

## 5. Comprobar que la GPU quedo realmente activaQue aparezca `CUDAExecutionProvider` en la lista no basta: la libreria puedefallar al cargar y caer a procesador sin avisar. Esta celda crea una sesion deverdad y reporta que proveedor quedo en uso.

In [ ]:
import numpy as np, onnxruntime as ortfrom open_image_models import create_detectorprint("Compilados:", ort.get_available_providers())detector = create_detector("rf-detr-nano-384-coco", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])activos = detector.model.get_providers()print("Realmente activos:", activos)print()if any("CUDA" in p or "Tensorrt" in p for p in activos):    print("GPU ACTIVA. Puedes seguir.")else:    print("LA GPU NO ESTA ACTIVA: el proceso correria en procesador y tardaria horas de mas.")    print("Revisa que el entorno tenga GPU y que hayas reiniciado tras la celda 3.")

## 6. Indicar donde estan tus videosCambia la ruta si tu carpeta se llama distinto.Si los videos te los compartieron, abre la carpeta compartida en Drive y usa**Organizar > Anadir acceso directo**, eligiendo Mi unidad. Asi aparece dentrode tu Drive montado.

In [ ]:
from pathlib import PathCARPETA_VIDEOS = "/content/drive/MyDrive/Cam PL"CARPETA_SALIDA = "/content/drive/MyDrive/informe_lastre"videos = sorted(Path(CARPETA_VIDEOS).glob("*.mp4"))print(f"Videos encontrados: {len(videos)}")for v in videos[:5]:    print("  ", v.name)if len(videos) > 5:    print(f"   ... y {len(videos) - 5} mas")

## 7. Verificar la zona de analisisEl poligono de la via de lastre esta calibrado para esta camara en su posicionactual. Esta celda dibuja la zona sobre un cuadro real para que confirmes quecubre el lastre y deja fuera la carretera principal.Si no coincide, edita `config/zona.json` y vuelve a ejecutar.

In [ ]:
from IPython.display import Image, display!python scripts/verificar_zona.py "{videos[0]}" --frame 2697display(Image("out/verificacion_zona_f2697.jpg", width=900))

## 8. Prueba con 3 videosMide cuanto tarda antes de comprometer horas. Multiplica ese tiempo por elnumero real de videos.La barra de avance va de 0 a 100 sobre el total de cuadros del lote, e incluyeuna estimacion del tiempo restante.

In [ ]:
!python scripts/procesar_lote.py \    "{CARPETA_VIDEOS}" \    --out-dir "{CARPETA_SALIDA}" \    --acelerador gpu \    --limite 3

## 9. Revisar el resultado de la pruebaAntes de lanzar el lote completo, mira lo que produjo.

In [ ]:
from openpyxl import load_workbooklibro = load_workbook(f"{CARPETA_SALIDA}/placas_lastre.xlsx")hoja = libro["Placas"]print(f"Vehiculos registrados: {hoja.max_row - 1}")print()for fila in hoja.iter_rows(min_row=1, max_row=min(11, hoja.max_row), values_only=True):    print("  ".join(str(c)[:18].ljust(18) for c in fila[:7]))print()print("--- Resumen ---")resumen = libro["Resumen"]for i in range(3, 20):    clave = resumen.cell(row=i, column=1).value    if clave:        print(f"  {clave}: {resumen.cell(row=i, column=2).value}")

## 10. Procesar todos los videosQuita `--limite` para el lote completo. Agrega `--solo-salidas` si unicamentete interesan los vehiculos que salen por el lastre.**Si Colab se desconecta no pierdes nada:** el avance se guarda al terminarcada video. Vuelve a ejecutar esta misma celda y continua donde quedo.

In [ ]:
!python scripts/procesar_lote.py \    "{CARPETA_VIDEOS}" \    --out-dir "{CARPETA_SALIDA}" \    --acelerador gpu

## 11. UtilidadesEjecuta solo la que necesites.

In [ ]:
# Regenerar el Excel con lo ya procesado, sin analizar mas video!python scripts/procesar_lote.py "{CARPETA_VIDEOS}" --out-dir "{CARPETA_SALIDA}" --solo-informe

In [ ]:
# Empezar de cero, descartando todo el avance guardado!python scripts/procesar_lote.py "{CARPETA_VIDEOS}" --out-dir "{CARPETA_SALIDA}" \    --acelerador gpu --reiniciar

In [ ]:
# Ver cuantos videos van procesadosimport jsonwith open(f"{CARPETA_SALIDA}/avance.json") as f:    avance = json.load(f)print(f"Videos procesados: {len(avance['videos'])}")for nombre, datos in sorted(avance["videos"].items()):    print(f"  {nombre}: {len(datos['filas'])} vehiculos")

## Que obtienesEn la carpeta de salida de tu Drive:- **`placas_lastre.xlsx`** con una fila por vehiculo y su foto incrustada al  lado. Verde es validado, amarillo pendiente de revision, rojo sin placa  legible. Una segunda hoja resume los totales.- **`recortes/`** con las imagenes sueltas por si necesitas ampliarlas.- **`avance.json`** con el estado del procesamiento.Revisa a mano las filas amarillas, y tambien las verdes con consenso bajo: esacolumna dice que tan disputada fue la lectura entre los distintos cuadros.